# 03 — Normalization, dimensionality reduction, clustering

Equivalent to `scripts/03_normalize_cluster.py`. Produces **Figure 2** and answers
**Research Question 1**. This is the slow step: 20–60 minutes.

## The one thing to understand here

`adata.X` gets overwritten repeatedly. After scaling it holds z-scores, which are
meaningless for differential expression — you cannot take a fold change of a
z-score. So before scaling we stash the log-normalized matrix in `adata.raw`.
Every DE call downstream must use `use_raw=True`.

**This is the single most common bug in student scRNA-seq pipelines.** It produces
plausible-looking numbers that are entirely wrong.

## Memory (16 GB machine)

Restart the kernel before running this notebook, and close other notebooks — a
stale kernel holding a copy of `adata` is the usual cause of an OOM kill here.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import config

sc.settings.verbosity = 3
sc.settings.figdir = config.FIG_DIR
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.logging.print_header()

In [ ]:
import gc
adata = sc.read_h5ad(config.H5AD_QC)
print(f'{adata.n_obs:,} cells x {adata.n_vars:,} genes')

## Normalize

A cell with 20,000 UMIs is not expressing more than one with 5,000 — it was
sequenced deeper. Scale every cell to the same total, then `log1p` to tame the long
right tail so a 2-fold change means the same thing at high and low expression.

The paper used **SCTransform** (regularized negative binomial) instead. That is a
genuinely different variance model and is part of why your cluster count will
differ from theirs. State it in Methods.

In [ ]:
adata.layers['counts'] = adata.X.copy()   # keep integers for pseudobulk later

sc.pp.normalize_total(adata, target_sum=config.TARGET_SUM)
sc.pp.log1p(adata)

adata.X = adata.X.astype('float32')   # halves memory, loses no real precision
adata.raw = adata                     # full genes, log-normalized -> DE uses this

## Highly variable genes

Most of ~13,000 genes are uninformative noise for clustering. `batch_key='sample'`
ranks genes by how *consistently* variable they are across samples, so one
anomalous sample cannot drive the selection.

In [ ]:
sc.pp.highly_variable_genes(
    adata, n_top_genes=config.N_TOP_GENES,
    batch_key=config.HVG_BATCH_KEY, flavor='seurat',
)
sc.pl.highly_variable_genes(adata)
print(int(adata.var['highly_variable'].sum()), 'HVGs selected')

Subsetting to HVGs **before** scaling is the key memory decision: `regress_out` and
`scale` on 2,000 genes take minutes, on 13,000 they can exhaust 16 GB.
`adata.raw` still holds all genes for DE.

In [ ]:
adata = adata[:, adata.var['highly_variable']].copy()
gc.collect()
print(f'{adata.n_obs:,} cells x {adata.n_vars:,} genes')

## Regress out technical covariates, then scale

Slowest cell in the project. If the kernel dies here, set `REGRESS_OUT = False` in
`config.py` — that is the largest single memory saving and the biology barely moves.

In [ ]:
if config.REGRESS_OUT:
    sc.pp.regress_out(adata, config.REGRESS_VARS)
else:
    print('Skipping regress_out (REGRESS_OUT=False)')

sc.pp.scale(adata, max_value=config.SCALE_MAX_VALUE)
gc.collect()

## PCA

Look at the elbow before accepting `N_PCS`. Components past the elbow are noise,
and including them blurs cluster boundaries.

In [ ]:
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True,
          random_state=config.RANDOM_SEED)
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)
print('config.N_PCS =', config.N_PCS)

## Neighbour graph and UMAP

In [ ]:
sc.pp.neighbors(adata, n_neighbors=config.N_NEIGHBORS, n_pcs=config.N_PCS,
                random_state=config.RANDOM_SEED)
sc.tl.umap(adata, random_state=config.RANDOM_SEED)

## Resolution sweep — the honest answer to RQ1

The paper justified resolution 0.8 by showing the cluster count plateaus there.
Reproducing that curve is a much stronger result than a single number, and it is
how you handle "we got 31, not 36" without either hiding it or tuning until it
matches.

In [ ]:
def run_leiden(adata, resolution, key):
    try:
        sc.tl.leiden(adata, resolution=resolution, key_added=key,
                     flavor='igraph', n_iterations=2, directed=False,
                     random_state=config.RANDOM_SEED)
    except (TypeError, ValueError):
        sc.tl.leiden(adata, resolution=resolution, key_added=key,
                     random_state=config.RANDOM_SEED)
    return adata.obs[key].nunique()

sweep = []
for res in config.RESOLUTION_SWEEP:
    n = run_leiden(adata, res, f'leiden_sweep_{res}')
    sweep.append({'resolution': res, 'n_clusters': n})
    print(f'resolution {res}: {n} clusters')

sweep_df = pd.DataFrame(sweep)
sweep_df

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(sweep_df['resolution'], sweep_df['n_clusters'], 'o-')
ax.axhline(config.TARGET_N_CLUSTERS, ls='--', c='crimson',
           label=f'paper: {config.TARGET_N_CLUSTERS}')
ax.set_xlabel('Leiden resolution'); ax.set_ylabel('clusters')
ax.legend(frameon=False); fig.tight_layout()
fig.savefig(config.FIG_DIR / '03_resolution_sweep.png', dpi=150)

## Primary clustering at the paper's resolution

In [ ]:
n_main = run_leiden(adata, config.LEIDEN_RESOLUTION, config.LEIDEN_KEY)
print(f'resolution {config.LEIDEN_RESOLUTION}: {n_main} clusters (paper: {config.TARGET_N_CLUSTERS})')

In [ ]:
sc.pl.umap(adata, color=[config.LEIDEN_KEY], legend_loc='on data', legend_fontsize=6)

## Is there a batch effect?

The paper reports no cluster was dominated by one sample. Even mixing across 8
samples is 12.5% each. A cluster above ~40% from one sample is a batch effect, not
a cell type — set `USE_HARMONY = True` in `config.py` and re-run.

In [ ]:
sc.pl.umap(adata, color=['sex', 'treatment', 'condition', 'sample'], ncols=2)

In [ ]:
comp = pd.crosstab(adata.obs[config.LEIDEN_KEY], adata.obs['sample'], normalize='index')
comp.max(axis=1).sort_values(ascending=False).head(10).round(3)

In [ ]:
sc.pl.umap(adata, color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'], ncols=3)

In [ ]:
adata.write(config.H5AD_CLUSTERED)
print('Wrote', config.H5AD_CLUSTERED)